# 01 — The Control Problem

**Chapter 2 teaching lab — §2.1**

> **Teaching outcome:** Separate model proposal from execution authority.

Flow: *concept → why it matters → diagram → read the implementation → expected behavior →
red-team round → playground → challenge.*

In [ ]:
# Setup: make harnesslab/ importable from any working directory, then load this lab's controls.
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "harnesslab").is_dir())
sys.path.insert(0, str(ROOT))

from harnesslab import Action, AutonomyLevel, Decision, EXECUTES, Reversibility
from harnesslab.authority import Controller, DenylistController, Harness, ProposedAction, Tool
from harnesslab.viz import decisions
from harnesslab.playground import authority_playground
from harnesslab.arena import NAIVE, SECURE, run_gauntlet
from harnesslab.viz import gauntlet_board, journey, show_source
from IPython.display import display

journey(1)

## Concept
A model produces **proposals**: text that describes an action. Nothing about a proposal grants the
right to carry it out. The **harness** is the boundary where a proposal either acquires execution
authority or does not. If model output flows straight into a tool, the model *is* the authority, and
every prompt injection, hallucinated argument, or misread instruction becomes an executed side effect.

## Why this matters
Most agent incidents are not "the model was wrong". They are "the model was wrong **and nothing stood
between the proposal and the side effect**". Locating the boundary explicitly is the first design act
of every later control in this chapter.

## Diagram
```mermaid
flowchart LR
  M[Model] -->|proposal| C{"Controller<br/>on the allowlist?"}
  subgraph H [Harness boundary]
    C
    P[PAUSED + audit]
  end
  C -->|yes| T[Tool] --> E((side effect))
  C -->|no| P
  classDef bad fill:#dc262622,stroke:#dc2626
  classDef ok fill:#16a34a22,stroke:#16a34a
  class P bad
  class T ok
```

## Read the implementation
The controller is **default deny**: only allowlisted actions run without a human. The harness audits
every proposal, executed or not.

In [ ]:
show_source(Controller, Harness)

## Experiment: the unsafe direct path

In [ ]:
tool = Tool()
proposal = ProposedAction("publish_report", {"report_id": "R-104"})
print("Unsafe direct path:", tool.execute(proposal))
assert tool.executed == ["publish_report"]  # the model alone decided to publish

## Experiment: the same proposals through a harness

In [ ]:
tool = Tool()
harness = Harness(Controller(), tool)
for p in [ProposedAction("search_reports", {"q": "Q3"}), ProposedAction("publish_report", {"report_id": "R-104"})]:
    print(harness.run(p))
decisions([(name, "allow" if status == "executed" else "deny", status) for name, status in harness.audit],
          title="Harness audit log")

## Expected behavior

In [ ]:
assert tool.executed == ["search_reports"]
assert harness.audit == [("search_reports", "executed"), ("publish_report", "paused")]
print("Proposal and execution are separated.")

## Red-team round (§2.1)
The same attacks run against two defense sets: **HarnessLab** (what this lab built) and
**Shortcuts** (the patterns the chapter warns about). Every attack must be blocked by HarnessLab.
The first draft of this lab used a denylist: `name != "publish_report"`. Watch what a near-miss name does to it.

In [ ]:
secure, shortcuts = run_gauntlet(SECURE, "2.1"), run_gauntlet(NAIVE, "2.1")
assert all(r.blocked for r in secure), [r.attack.name for r in secure if not r.blocked]
gauntlet_board(secure, shortcuts, title="Gauntlet · §2.1")

## Playground
Type any action name. Try `publish_report`, `PUBLISH_REPORT`, `search_reports `, `draft_report`.

In [ ]:
authority_playground()

## Challenge
1. Add `search_reports` and `delete_report` tools. Which belong on the allowlist? Justify using
   reversibility, not convenience.
2. Extend the audit record with arguments and a timestamp. What would an incident reviewer need that is
   still missing?
3. The controller sees only the action *name*. Construct a proposal that is dangerous only because of
   its *arguments*. How would you close that gap? (Notebook 04 returns to this.)